
# Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
model = ChatOpenAI(
    model="gpt-4.1",
    api_key = openai_api_key,
    base_url = openai_base_url
)
model

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002007EF24D70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002007EF25940>, root_client=<openai.OpenAI object at 0x000002007EA657F0>, root_async_client=<openai.AsyncOpenAI object at 0x000002007EF256A0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://yibuapi.com/v1')

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie out of 10")


In [5]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002007EF24D70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002007EF25940>, root_client=<openai.OpenAI object at 0x000002007EA657F0>, root_async_client=<openai.AsyncOpenAI object at 0x000002007EF256A0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://yibuapi.com/v1'), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structure

In [6]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception** is a science fiction action thriller film released in 2010. Here are its key details:\n\n**Director & Writer:**\n- Christopher Nolan\n\n**Main Cast:**\n- Leonardo DiCaprio as Dom Cobb\n- Joseph Gordon-Levitt as Arthur\n- Ellen Page (now Elliot Page) as Ariadne\n- Tom Hardy as Eames\n- Ken Watanabe as Saito\n- Cillian Murphy as Robert Fischer\n- Marion Cotillard as Mal\n- Michael Caine as Professor Miles\n\n**Plot Summary:**\nThe film centers around Dom Cobb (Leonardo DiCaprio), a skilled thief and “extractor” who specializes in stealing secrets from people’s subconscious during their dreams. Cobb is offered a chance to have his criminal record erased if he can successfully perform "inception": planting an idea in someone’s mind rather than stealing one. Cobb and his team, including Arthur, Ariadne, and Eames, embark on a complex mission involving multiple layers of dreams within dreams. As they navigate the psychological landscape, Cobb must confront me

In [7]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide information about the movie Inception.")
response

{'raw': AIMessage(content='{"director":"Frank Darabont","rating":9.3,"title":"The Shawshank Redemption","year":1994}', additional_kwargs={'parsed': Movie(title='The Shawshank Redemption', year=1994, director='Frank Darabont', rating=9.3), 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 115, 'total_tokens': 143, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1', 'system_fingerprint': 'fp_e05894342c', 'id': 'chatcmpl-DKI7qMDRq4qvP4VBbITlt8Py8FaXD', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cfa76-a640-7dc2-b3a0-87a153c171e0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 115, 'output_tokens': 28, 'total_tokens': 143, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'

## Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    age: int = Field(..., description="The age of the actor")
    role: str = Field(..., description="The role of the actor")

class MovieDetails(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    cast: list[Actor] = Field(..., description="The actors in the movie")
    genres: list[str] = Field(..., description="The genres of the movie")
    budget: float | None = Field(None, description="The budget of the movie")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide information about the movie Inception.")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', age=36, role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', age=29, role='Arthur'), Actor(name='Ellen Page', age=35, role='Ariadne'), Actor(name='Tom Hardy', age=38, role='Eames'), Actor(name='Ken Watanabe', age=39, role='Saito'), Actor(name='Cillian Murphy', age=61, role='Robert Fischer'), Actor(name='Michael Caine', age=70, role='Professor Miles')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160000000.0)

# TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [12]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide information about the movie Inception.")
response

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

In [13]:

class Actor(TypedDict):
    name: Annotated[str, ..., "The name of the actor"]
    age: Annotated[int, ..., "The age of the actor"]
    role: Annotated[str, ..., "The role of the actor"]

class MovieDetails(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    cast: Annotated[list[Actor], ..., "The actors in the movie"]
    genres: Annotated[list[str], ..., "The genres of the movie"]
    budget: Annotated[float | None, ..., "The budget of the movie"]

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide information about the movie Inception.")
response

{'title': 'Inception',
 'year': 2010,
 'genres': ['Science Fiction', 'Action', 'Thriller'],
 'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb', 'age': 35},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur', 'age': 29},
  {'name': 'Elliot Page', 'role': 'Ariadne', 'age': 23},
  {'name': 'Tom Hardy', 'role': 'Eames', 'age': 33},
  {'name': 'Ken Watanabe', 'role': 'Saito', 'age': 51},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer', 'age': 34},
  {'name': 'Marion Cotillard', 'role': 'Mal Cobb', 'age': 34},
  {'name': 'Michael Caine', 'role': 'Miles', 'age': 77}]}

In [14]:
model.profile

{'max_input_tokens': 1047576,
 'max_output_tokens': 32768,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'structured_output': True,
 'image_url_inputs': True,
 'pdf_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_choice': True}

# DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [16]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a persion."""
    name: str = Field(..., description="The name of the contact")
    email: str = Field(..., description="The email address of the contact")
    phone: str = Field(..., description="The phone number of the contact")

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact information from John Doe, john.doe@example.com, 555-1234"}]
})

result

{'messages': [HumanMessage(content='Extract contact information from John Doe, john.doe@example.com, 555-1234', additional_kwargs={}, response_metadata={}, id='4f04a636-76af-4328-bb1c-32e134e12118'),
  AIMessage(content='{"name":"John Doe","email":"john.doe@example.com","phone":"555-1234"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 483, 'prompt_tokens': 110, 'total_tokens': 593, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'input_tokens': 0, 'input_tokens_details': None, 'output_tokens': 0}, 'model_provider': 'openai', 'model_name': 'gpt-5', 'system_fingerprint': None, 'id': 'chatcmpl-DKIVtGmMVTWD7yfbNXdvvwexeGosO', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cfa8d-796a-7452-b55d-ef9afe198686-0', tool_calls=[], invalid_tool_calls=[], usage_metad

In [19]:
result["structured_response"]

ContactInfo(name='John Doe', email='john.doe@example.com', phone='555-1234')

In [20]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a persion."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact information from John Doe, john.doe@example.com, 555-1234"}]
})

result["structured_response"]

{'messages': [HumanMessage(content='Extract contact information from John Doe, john.doe@example.com, 555-1234', additional_kwargs={}, response_metadata={}, id='0e12fd24-d949-40d1-9036-d1fa0bdcb2fa'),
  AIMessage(content='{"phone":"555-1234","email":"john.doe@example.com","name":"John Doe"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1571, 'prompt_tokens': 84, 'total_tokens': 1655, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1536, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'input_tokens': 0, 'input_tokens_details': None, 'output_tokens': 0}, 'model_provider': 'openai', 'model_name': 'gpt-5', 'system_fingerprint': None, 'id': 'chatcmpl-DKIXH8wVmyDXP2WkOQ4dJCXKTzbH3', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cfa8e-cdba-7740-8b5a-0b711d370363-0', tool_calls=[], invalid_tool_calls=[], usage_met

In [21]:
# DataClass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a persion."""
    name: str # The name of person
    email: str # The email address of the contact
    phone: str # The phone number of the contact

agent = create_agent(
    model = "gpt-5",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact information from John Doe, john.doe@example.com, 555-1234"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john.doe@example.com', phone='555-1234')